In [1]:
import torch
import torch.nn.functional as F
import requests

In [2]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

print(words[:10])
len(words)

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


32033

In [3]:
def stoi(s):
    return {ch: i for i, ch in enumerate(".abcdefghijklmnopqrstuvwxyz")}[s]
def getXY(word, block_size):
    X = []
    Y = []
    context = [0] * block_size
    for ch in word + ".":
        X.append(context)
        Y.append(stoi(ch))
        ix = stoi(ch)
        context = context[1:] + [ix]
        print(context)
    return torch.tensor(X), torch.tensor(Y)

X, Y = getXY("hello", 8)



[0, 0, 0, 0, 0, 0, 0, 8]
[0, 0, 0, 0, 0, 0, 8, 5]
[0, 0, 0, 0, 0, 8, 5, 12]
[0, 0, 0, 0, 8, 5, 12, 12]
[0, 0, 0, 8, 5, 12, 12, 15]
[0, 0, 8, 5, 12, 12, 15, 0]


In [4]:
print(X.shape)
print(X)
print(Y.shape)
print(Y)

torch.Size([6, 8])
tensor([[ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  8],
        [ 0,  0,  0,  0,  0,  0,  8,  5],
        [ 0,  0,  0,  0,  0,  8,  5, 12],
        [ 0,  0,  0,  0,  8,  5, 12, 12],
        [ 0,  0,  0,  8,  5, 12, 12, 15]])
torch.Size([6])
tensor([ 8,  5, 12, 12, 15,  0])


In [5]:
C = torch.randn((27, 2), requires_grad=True)
hot = F.one_hot(Y, num_classes=27).float()
hot @ C

tensor([[ 0.8106,  0.9699],
        [ 0.5894, -0.5682],
        [ 0.2334, -1.6092],
        [ 0.2334, -1.6092],
        [ 0.6400, -0.7467],
        [-1.4492,  1.2366]], grad_fn=<MmBackward0>)

In [6]:
emb = C[X]
print(emb.shape)
print(emb)

torch.Size([6, 8, 2])
tensor([[[-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366]],

        [[-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [ 0.8106,  0.9699]],

        [[-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [ 0.8106,  0.9699],
         [ 0.5894, -0.5682]],

        [[-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [-1.4492,  1.2366],
         [ 0.8106,  0.9699],
         [ 0.5894, -0.5682],
         [ 0.2334, -1.6092]],

        [[-1.4492,  1.2366],
         [-1.

In [7]:
W1 = torch.randn((16, 100), requires_grad=True)
b1 = torch.randn(100, requires_grad=True)
h = emb.view(-1, 16) @ W1 + b1
print(h.shape)


torch.Size([6, 100])


In [30]:
W2 = torch.randn((100, 27), requires_grad=True)
b2 = torch.randn(27, requires_grad=True)
logits = h @ W2 + b2
print(logits.shape)
optimizer = torch.optim.Adam([W1, b1, W2, b2, C], lr=0.01)

torch.Size([6, 27])


In [63]:
#overfit test
hot = F.one_hot(Y, num_classes=27).float()
emb = C[X]
h = torch.tanh(emb.view(-1, 16) @ W1 + b1)
logits = h @ W2 + b2
#backwords
loss = F.cross_entropy(logits, Y)
loss.backward()
#forward

optimizer.step()
print(loss)
optimizer.zero_grad()


tensor(0.0015, grad_fn=<NllLossBackward0>)
